# Datos de práctica · Mercado Sur

Este archivo contiene **solo los datos** del desafío: tres tablas ficticias de una organización que vende por web y en tienda. Es **autocontenido** (no necesita ningún otro archivo) y los datos se generan desde código, así que todos los equipos trabajan con exactamente los mismos.

El pedido de tu equipo y sus definiciones los recibís aparte. Todo el análisis lo escribís vos, en celdas nuevas debajo de la última.

**Cómo se usa:** corré las celdas de arriba hacia abajo con `Shift` + `Enter`. Después, escribí tus consultas en celdas que empiecen con `%%sql`: el resultado se muestra como tabla y se guarda en la variable que nombres (por ejemplo, `%%sql mi_tabla` lo guarda en `mi_tabla`).

## Preparación

In [ ]:
# Este archivo es autocontenido: instala solo lo que le falte (se corre una sola vez)
import importlib.util
import subprocess
import sys

LIBRERIAS = ("numpy", "pandas", "duckdb", "matplotlib")
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    for extra in ([], ["--user"], ["--break-system-packages"]):          # si un intento falla, prueba el siguiente
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra, *faltan], capture_output=True, text=True)
        if r.returncode == 0:
            break
    else:
        print("⚠️ No se pudo instalar solo. Corré en una celda:  %pip install " + " ".join(faltan))
        print(r.stderr[-300:])
    importlib.invalidate_caches()
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
print("✅ Librerías listas" if not faltan else f"⚠️ Todavía faltan: {faltan}")

In [ ]:
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display

con = duckdb.connect()          # el motor de SQL, dentro del notebook

def _sql(line, cell):
    """Las celdas que empiezan con %%sql ejecutan SQL. El resultado se guarda en la variable que nombres (por ejemplo: %%sql mi_tabla)."""
    nombre = line.strip().split()[0] if line.strip() else "resultado"
    try:
        df = con.sql(cell).df()
    except Exception as e:
        print("Error de SQL:", e)
        return
    get_ipython().user_ns[nombre] = df
    print(f"{len(df):,} filas · {len(df.columns)} columnas · guardado en «{nombre}»")
    display(df if len(df) <= 40 else df.head(20))

get_ipython().register_magic_function(_sql, "cell", "sql")
pd.options.display.float_format = lambda x: f"{x:,.2f}" if (abs(x) >= 1 or x == 0) else f"{x:.4f}"
print("pandas:", pd.__version__, "· duckdb:", duckdb.__version__)

## Paso 1 · Armamos los datos de Mercado Sur

No cargamos ningún archivo: **los generamos con código**, así todos tenemos exactamente los mismos datos. Son **tres tablas relacionadas**:

```
clientes ──(id_cliente)──▶ ventas ◀──(id_producto)── productos
quién compra                qué ocurrió               qué se vende
```

| Tabla | Una fila representa… |
| --- | --- |
| `clientes` | un cliente |
| `productos` | un producto del catálogo |
| `ventas` | **una operación de venta** (completada, devuelta o cancelada) |

### 1.1 · La semilla y los parámetros

Casi todo lo que sigue es **un sorteo** (precios, fechas, canales…). Para que el sorteo sea siempre el mismo usamos una **semilla**: con la misma semilla, cualquiera que corra este código obtiene exactamente los mismos datos.

In [ ]:
SEMILLA = 2025                                 # el «número de sorteo»
rng = np.random.default_rng(SEMILLA)           # el generador de sorteos
N_CLIENTES = 1500
N_OPS = 12000                                  # operaciones que se sortean (algunas se descartan más adelante)
print("Listo: vamos a generar", N_CLIENTES, "clientes y", N_OPS, "operaciones")

### 1.2 · La tabla `productos`

Cinco categorías, **ocho productos por categoría**. Cada categoría tiene un rango de precios (la electrónica es cara; los alimentos, baratos), y de ahí sale el `precio_lista` de cada producto.

In [ ]:
CATEGORIAS = {"Electrónica": (150, 900, 0.15), "Hogar": (30, 300, 0.20), "Indumentaria": (20, 150, 0.20),
              "Deportes": (25, 250, 0.15), "Alimentos": (3, 40, 0.30)}   # (precio mínimo, precio máximo, peso en las ventas)
filas, n = [], 1
for cat, (lo, hi, _) in CATEGORIAS.items():
    for _ in range(8):
        filas.append({"id_producto": f"P{n:03d}", "categoria": cat, "nombre": f"{cat} {n:02d}",
                      "precio_lista": round(float(rng.uniform(lo, hi)), 2)})
        n += 1
productos = pd.DataFrame(filas)
productos.head()

### 1.3 · La tabla `clientes`

Cada cliente tiene una **región** (Centro, Norte, Sur, Litoral o Cuyo) y una **fecha de alta**.

In [ ]:
REGIONES = ["Centro", "Norte", "Sur", "Litoral", "Cuyo"]
clientes = pd.DataFrame({
    "id_cliente": [f"C{i:04d}" for i in range(1, N_CLIENTES + 1)],
    "region": rng.choice(REGIONES, size=N_CLIENTES, p=[0.30, 0.20, 0.18, 0.17, 0.15]),
    "fecha_alta": pd.to_datetime("2023-01-01") + pd.to_timedelta(rng.integers(0, 900, N_CLIENTES), unit="D"),
})
clientes["fecha_alta"] = clientes["fecha_alta"].dt.strftime("%Y-%m-%d")
clientes.head()

### 1.4 · La tabla `ventas`: quién compró qué, cuándo y por qué canal

Ahora las **operaciones**. Para cada una sorteamos la **fecha** (durante 2025), **quién** compró, por qué **canal** (web o tienda) y **qué producto**. Después, las **unidades**, el **precio unitario** (cerca del precio de lista) y el **descuento**.

Para armarla usamos columnas auxiliares (`categoria`, `precio_lista`, `region`) que al final descartamos.

In [ ]:
fechas = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, N_OPS), unit="D")
ventas = pd.DataFrame({
    "id_venta": [f"V{i:06d}" for i in range(1, N_OPS + 1)],
    "fecha": fechas,
    "id_cliente": rng.choice(clientes["id_cliente"], size=N_OPS),
    "canal": rng.choice(["web", "tienda"], size=N_OPS, p=[0.55, 0.45]),
})
pesos = np.array([v[2] for v in CATEGORIAS.values()])
cats = np.array(list(CATEGORIAS.keys()))[rng.choice(len(CATEGORIAS), size=N_OPS, p=pesos / pesos.sum())]
ventas["id_producto"] = [rng.choice(productos.loc[productos["categoria"] == c, "id_producto"]) for c in cats]
ventas = ventas.merge(productos[["id_producto", "categoria", "precio_lista"]], on="id_producto", how="left")
ventas = ventas.merge(clientes[["id_cliente", "region"]], on="id_cliente", how="left")
ventas["unidades"] = np.where(ventas["categoria"] == "Alimentos", rng.integers(1, 9, N_OPS), rng.integers(1, 5, N_OPS))
ventas["precio_unitario"] = (ventas["precio_lista"] * rng.uniform(0.97, 1.05, N_OPS)).round(2)
ventas["descuento_pct"] = rng.choice([0.0, 0.05, 0.10, 0.15, 0.20], size=N_OPS, p=[0.45, 0.15, 0.20, 0.12, 0.08])
print(f"{len(ventas):,} operaciones sorteadas")

### 1.5 · El estado de cada operación

Cada operación termina en uno de tres **estados**: `completada`, `devuelta` o `cancelada`. Casi todas se completan. Se genera con una regla interna que **no vamos a mostrar**: descubrir qué pasa es parte del trabajo de análisis.

In [ ]:
s2 = ventas["fecha"].dt.month > 6
seg = (ventas["canal"] == "web") & (ventas["region"] == "Litoral") & (ventas["categoria"] == "Electrónica") & s2
seg_indum = (ventas["canal"] == "web") & (ventas["categoria"] == "Indumentaria") & s2
p_comp, p_dev, p_can = np.full(N_OPS, 0.88), np.full(N_OPS, 0.07), np.full(N_OPS, 0.05)
p_comp[seg], p_dev[seg], p_can[seg] = 0.64, 0.30, 0.06
p_comp[seg_indum], p_dev[seg_indum], p_can[seg_indum] = 0.81, 0.13, 0.06
u = rng.random(N_OPS)
ventas["estado"] = np.where(u < p_comp, "completada", np.where(u < p_comp + p_dev, "devuelta", "cancelada"))
ventas = ventas.loc[~(seg & (rng.random(N_OPS) < 0.38))].copy()
print(f"Estados asignados · operaciones: {len(ventas):,}")

### 1.6 · La satisfacción (con faltantes)

Algunos clientes califican la compra de **1 a 5**. Pero **no todos responden**: la satisfacción puede faltar, y falta más en las operaciones canceladas y devueltas. Los faltantes **se analizan, no se reemplazan**.

In [ ]:
n_v = len(ventas)
sat = rng.choice([1, 2, 3, 4, 5], size=n_v, p=[0.05, 0.08, 0.17, 0.35, 0.35]).astype(float)
dev = (ventas["estado"] == "devuelta").to_numpy()
can = (ventas["estado"] == "cancelada").to_numpy()
sat[dev] = rng.choice([1, 2, 3], size=dev.sum(), p=[0.5, 0.3, 0.2])               # las devueltas califican peor
sat[rng.random(n_v) < np.where(can, 0.80, np.where(dev, 0.25, 0.12))] = np.nan     # y los faltantes
ventas["satisfaccion"] = sat

ventas = ventas.sort_values("fecha").reset_index(drop=True)                          # ordenadas por fecha
ventas["id_venta"] = [f"V{i:06d}" for i in range(1, len(ventas) + 1)]
ventas["fecha"] = ventas["fecha"].dt.strftime("%Y-%m-%d")
ventas = ventas[["id_venta", "fecha", "id_cliente", "id_producto", "canal", "unidades",
                 "precio_unitario", "descuento_pct", "estado", "satisfaccion"]]       # descartamos las columnas auxiliares
ventas.head()

Esto es lo que significa cada columna de `ventas`:

In [ ]:
pd.DataFrame({
    "columna": ["id_venta", "fecha", "id_cliente", "id_producto", "canal", "unidades", "precio_unitario",
                "descuento_pct", "estado", "satisfaccion"],
    "qué es": ["Identificador de la operación", "Día de la operación (2025)", "Quién compró (une con `clientes`)",
               "Qué se vendió (une con `productos`)", "web o tienda", "Cuántas unidades",
               "Precio por unidad en esa operación", "Descuento, como fracción de 0 a 1 (0.10 = 10 %)",
               "completada, devuelta o cancelada", "Encuesta de 1 a 5 (puede faltar)"]})

Por último, le contamos a DuckDB cuáles son las tres tablas, para poder consultarlas con SQL (las fechas pasan a ser fechas de verdad):

In [ ]:
con.register("clientes", clientes.assign(fecha_alta=pd.to_datetime(clientes["fecha_alta"])))
con.register("productos", productos)
con.register("ventas", ventas.assign(fecha=pd.to_datetime(ventas["fecha"])))
print({t: con.sql(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("clientes", "productos", "ventas")})

**Listo.** Quedaron registradas `clientes`, `productos` y `ventas`. Desde acá, el trabajo es de tu equipo: **escribí tus consultas debajo**.